<a href="https://colab.research.google.com/github/celinekm/CN/blob/main/Tarefa_Interpolacao_Erros_Resolvida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**NOME: CELINE KAZUMI MIYAJIMA RA: 176209**

# Tarefa investigativa: erros em métodos de interpolação
## Reconstrução da temperatura em uma barra aquecida

**Resolução completa das etapas 1 a 4.** O notebook compara Lagrange global, spline linear, spline quadrática e spline cúbica natural, analisa a posição e a quantidade de sensores, mede tempos de execução e investiga o efeito do ruído.

O perfil de referência é:
$$
T(x)=18+2.4\frac{x}{6}+2.5\sin\left(\pi\frac{x}{6}\right),\quad 0\leq x\leq6.
$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from scipy.interpolate import BarycentricInterpolator, interp1d, CubicSpline

np.set_printoptions(precision=5, suppress=True)
plt.rcParams["figure.figsize"] = (10, 5)

L = 6.0
x_min, x_max = 0.0, L
x_fino = np.linspace(x_min, x_max, 2001)

def temperatura_real(x):
    x = np.asarray(x, dtype=float)
    xi = x / L
    return 18 + 2.4*xi + 2.5*np.sin(np.pi*xi)

T_fino = temperatura_real(x_fino)
print(f"T(0) = {temperatura_real(0):.1f} °C")
print(f"T(L) = {temperatura_real(L):.1f} °C")

plt.plot(x_fino, T_fino, label="Temperatura de referência")
plt.xlabel("Posição x (m)")
plt.ylabel("Temperatura (°C)")
plt.title("Perfil analítico de temperatura na barra")
plt.grid(alpha=.3)
plt.legend()
plt.show()


## Funções auxiliares

A spline quadrática é construída por trechos $S_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2$, impondo interpolação nos extremos de cada intervalo, continuidade da primeira derivada e condição $c_0=0$. A spline cúbica é natural (segunda derivada nula nas extremidades).

In [ ]:
def spline_quadratica_coef(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    n = len(x)-1
    if n < 1 or len(y) != len(x) or np.any(np.diff(x) <= 0):
        raise ValueError("x deve ser crescente e ter o mesmo tamanho de y.")
    A = np.zeros((3*n, 3*n))
    rhs = np.zeros(3*n)
    row = 0

    for i in range(n):
        A[row, i] = 1
        rhs[row] = y[i]
        row += 1

    for i in range(n):
        h = x[i+1]-x[i]
        A[row, i] = 1
        A[row, n+i] = h
        A[row, 2*n+i] = h*h
        rhs[row] = y[i+1]
        row += 1

    for i in range(n-1):
        h = x[i+1]-x[i]
        A[row, n+i] = 1
        A[row, 2*n+i] = 2*h
        A[row, n+i+1] = -1
        row += 1

    A[row, 2*n] = 1
    coef = np.linalg.solve(A, rhs)
    return coef[:n], coef[n:2*n], coef[2*n:]

def spline_quadratica_avalia(x, coef, xq):
    a, b, c = coef
    xq = np.asarray(xq, dtype=float)
    idx = np.clip(np.searchsorted(x, xq, side="right")-1, 0, len(a)-1)
    t = xq-x[idx]
    return a[idx] + b[idx]*t + c[idx]*t**2

def criar_metodos(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    lagrange = BarycentricInterpolator(x, y)
    linear = interp1d(x, y, kind="linear", bounds_error=True)
    quad_coef = spline_quadratica_coef(x, y)
    quadratica = lambda z: spline_quadratica_avalia(x, quad_coef, z)
    cubica = CubicSpline(x, y, bc_type="natural")
    return {
        "Lagrange global": lagrange,
        "Spline linear": linear,
        "Spline quadrática": quadratica,
        "Spline cúbica natural": cubica
    }

def calcular_metricas(metodos, x_teste, y_verdadeiro):
    linhas = []
    for nome, metodo in metodos.items():
        erro = np.asarray(metodo(x_teste))-np.asarray(y_verdadeiro)
        linhas.append({
            "Método": nome,
            "Erro máximo (°C)": np.max(np.abs(erro)),
            "MAE (°C)": np.mean(np.abs(erro)),
            "RMSE (°C)": np.sqrt(np.mean(erro**2))
        })
    return pd.DataFrame(linhas).sort_values("RMSE (°C)").reset_index(drop=True)

print("Funções auxiliares definidas.")


## Etapa 1 — Interpolação com dados exatos

São utilizados sete sensores igualmente espaçados, incluindo as extremidades. Calculamos as métricas em uma malha fina e comparamos as curvas e os erros assinados.

In [ ]:
x_nos_7 = np.linspace(x_min, x_max, 7)
y_nos_7 = temperatura_real(x_nos_7)
metodos_7 = criar_metodos(x_nos_7, y_nos_7)

metricas_7 = calcular_metricas(metodos_7, x_fino, T_fino)
display(metricas_7.round(7))

plt.plot(x_fino, T_fino, "k--", linewidth=2.5, label="Referência")
for nome, metodo in metodos_7.items():
    plt.plot(x_fino, metodo(x_fino), label=nome)
plt.scatter(x_nos_7, y_nos_7, color="black", zorder=5, label="Sensores")
plt.xlabel("Posição x (m)")
plt.ylabel("Temperatura (°C)")
plt.title("Etapa 1 — Comparação dos métodos com 7 sensores exatos")
plt.grid(alpha=.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
for nome, metodo in metodos_7.items():
    erro = metodo(x_fino)-T_fino
    plt.plot(x_fino, erro, label=nome)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Posição x (m)")
plt.ylabel("Erro assinado (°C)")
plt.title("Etapa 1 — Erro dos métodos de interpolação")
plt.grid(alpha=.3)
plt.legend()
plt.show()


### Interpretação da etapa 1

Os quatro métodos passam pelos valores dos sensores, mas diferem entre os nós. O Lagrange utiliza um único polinômio para todo o intervalo, enquanto as splines trabalham localmente. O erro máximo representa o maior desvio absoluto observado; MAE e RMSE resumem o erro ao longo da barra. Para este perfil suave, a spline cúbica tende a representar bem a curvatura sem as oscilações que podem surgir em polinômios globais de grau elevado.

## Etapa 2 — Número e posição dos sensores

Comparamos Lagrange com nós igualmente espaçados e com nós de Chebyshev-Lobatto para $N=5,9,13,17$.

In [ ]:
def nos_chebyshev_lobatto(a, b, N):
    k = np.arange(N)
    z = np.cos(k*np.pi/(N-1))
    return np.sort((a+b)/2 + (b-a)*z/2)

resultados_nos = []
for N in [5, 9, 13, 17]:
    configuracoes = {
        "Equiespaçados": np.linspace(x_min, x_max, N),
        "Chebyshev-Lobatto": nos_chebyshev_lobatto(x_min, x_max, N)
    }
    for distribuicao, x_nos in configuracoes.items():
        y_nos = temperatura_real(x_nos)
        interpolador = BarycentricInterpolator(x_nos, y_nos)
        erro = interpolador(x_fino)-T_fino
        resultados_nos.append({
            "N": N,
            "Distribuição": distribuicao,
            "Erro máximo (°C)": np.max(np.abs(erro)),
            "RMSE (°C)": np.sqrt(np.mean(erro**2))
        })

tabela_nos = pd.DataFrame(resultados_nos)
display(tabela_nos.round(9))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for distribuicao, grupo in tabela_nos.groupby("Distribuição"):
    axes[0].plot(grupo["N"], grupo["Erro máximo (°C)"], marker="o", label=distribuicao)
    axes[1].plot(grupo["N"], grupo["RMSE (°C)"], marker="o", label=distribuicao)
axes[0].set_title("Erro máximo")
axes[1].set_title("RMSE")
for ax in axes:
    ax.set_xlabel("Número de sensores (N)")
    ax.set_ylabel("Erro (°C)")
    ax.set_yscale("log")
    ax.grid(alpha=.3)
    ax.legend()
plt.suptitle("Etapa 2 — Influência da quantidade e posição dos sensores")
plt.tight_layout()
plt.show()


### Interpretação da etapa 2

O aumento de sensores geralmente melhora a reconstrução porque fornece mais informações sobre o perfil. Entretanto, no Lagrange global, muitos nós equiespaçados podem provocar oscilações, principalmente nas extremidades (fenômeno de Runge). Os nós de Chebyshev-Lobatto concentram pontos nas bordas e frequentemente reduzem esse efeito. A comparação é específica para o perfil suave analisado: não existe garantia de que a mesma distribuição seja sempre superior para qualquer função.

## Etapa 3 — Estudo de caso e decisão técnica

Agora variamos $N$ de 4 a 40 e comparamos Lagrange equiespaçado, Lagrange com Chebyshev e spline cúbica natural. Também medimos separadamente o tempo de construção e de avaliação. Os tempos dependem do computador e da execução do Colab.

In [ ]:
Ns = range(4, 41)
resultados_tempo = []

for N in Ns:
    configuracoes = {
        "Lagrange equiespaçado": np.linspace(x_min, x_max, N),
        "Lagrange Chebyshev": nos_chebyshev_lobatto(x_min, x_max, N),
        "Spline cúbica natural": np.linspace(x_min, x_max, N)
    }

    for nome, x_nos in configuracoes.items():
        y_nos = temperatura_real(x_nos)

        inicio = time.perf_counter()
        if nome == "Spline cúbica natural":
            interpolador = CubicSpline(x_nos, y_nos, bc_type="natural")
        else:
            interpolador = BarycentricInterpolator(x_nos, y_nos)
        tempo_construcao = time.perf_counter()-inicio

        inicio = time.perf_counter()
        valores = interpolador(x_fino)
        tempo_avaliacao = time.perf_counter()-inicio

        erro = valores-T_fino
        resultados_tempo.append({
            "N": N,
            "Método": nome,
            "Erro máximo (°C)": np.max(np.abs(erro)),
            "RMSE (°C)": np.sqrt(np.mean(erro**2)),
            "Construção (ms)": tempo_construcao*1000,
            "Avaliação (ms)": tempo_avaliacao*1000
        })

tabela_etapa3 = pd.DataFrame(resultados_tempo)
display(tabela_etapa3.head(12).round(7))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for nome, grupo in tabela_etapa3.groupby("Método"):
    axes[0].plot(grupo["N"], grupo["Erro máximo (°C)"], label=nome)
    axes[1].plot(grupo["N"], grupo["RMSE (°C)"], label=nome)
for ax in axes:
    ax.set_yscale("log")
    ax.set_xlabel("Número de sensores (N)")
    ax.set_ylabel("Erro (°C)")
    ax.grid(alpha=.3)
    ax.legend()
axes[0].set_title("Erro máximo em função de N")
axes[1].set_title("RMSE em função de N")
plt.suptitle("Etapa 3 — Precisão dos métodos")
plt.tight_layout()
plt.show()

tempos_medios = tabela_etapa3.groupby("Método")[["Construção (ms)", "Avaliação (ms)"]].mean()
display(tempos_medios.round(5))

fig, ax = plt.subplots(figsize=(9, 5))
tempos_medios.plot(kind="bar", ax=ax)
ax.set_ylabel("Tempo médio (ms)")
ax.set_title("Tempo médio de construção e avaliação")
ax.grid(axis="y", alpha=.3)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Interpretação da etapa 3

Não há necessariamente um método superior para todos os valores de $N$. O Lagrange pode apresentar alta precisão com poucos nós para funções suaves, mas sua estabilidade pode piorar com muitos nós equiespaçados. Chebyshev tende a controlar oscilações nas extremidades. A spline cúbica é local e costuma apresentar comportamento estável quando se aumenta a quantidade de sensores. Em geral, o custo de construção e avaliação cresce com o número de nós, mas os tempos medidos são pequenos e variam conforme o ambiente. A decisão deve considerar simultaneamente erro, estabilidade, distribuição dos sensores e custo computacional.

## Etapa 4 — Ruído de medição e recomendação técnica

Simulamos ruído gaussiano nas leituras dos sete sensores. A semente aleatória é fixada para tornar o experimento reproduzível. Como a interpolação passa exatamente pelos dados, ela também reproduz o ruído nos nós; por isso, uma curva muito flexível pode incorporar as oscilações da medição.

In [ ]:
rng = np.random.default_rng(42)
x_ruido = np.linspace(x_min, x_max, 7)
y_exato = temperatura_real(x_ruido)
sigma = 0.20  # desvio-padrão do ruído, em °C
y_ruidoso = y_exato + rng.normal(0, sigma, size=len(x_ruido))

metodos_ruidosos = criar_metodos(x_ruido, y_ruidoso)
metricas_ruido = calcular_metricas(metodos_ruidosos, x_fino, T_fino)
display(metricas_ruido.round(7))

plt.plot(x_fino, T_fino, "k--", linewidth=2.5, label="Referência verdadeira")
for nome, metodo in metodos_ruidosos.items():
    plt.plot(x_fino, metodo(x_fino), label=nome)
plt.scatter(x_ruido, y_ruidoso, color="red", zorder=5, label="Sensores com ruído")
plt.xlabel("Posição x (m)")
plt.ylabel("Temperatura (°C)")
plt.title(f"Etapa 4 — Interpolação com ruído (sigma = {sigma:.2f} °C)")
plt.grid(alpha=.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 5))
for nome, metodo in metodos_ruidosos.items():
    plt.plot(x_fino, metodo(x_fino)-T_fino, label=nome)
plt.axhline(0, color="black", linewidth=1)
plt.xlabel("Posição x (m)")
plt.ylabel("Erro assinado (°C)")
plt.title("Erros com medições ruidosas")
plt.grid(alpha=.3)
plt.legend()
plt.show()


### Recomendação técnica final

1. **Poucos sensores exatos:** escolheria inicialmente a spline cúbica natural ou Lagrange, comparando o RMSE e o erro máximo. Para poucos nós e um perfil suave, Lagrange pode funcionar bem; a spline cúbica oferece uma representação local e tende a ser mais estável.
2. **Aumento do número de sensores:** evitaria aumentar indiscriminadamente o grau do polinômio global com nós equiespaçados. Testaria nós de Chebyshev quando Lagrange fosse necessário e manteria a spline cúbica como alternativa robusta.
3. **Sensores com ruído:** não recomendaria interpolar diretamente todos os valores ruidosos como se fossem exatos. A interpolação reproduz o ruído nos nós. Usaria repetição de medidas, filtragem ou regressão/smoothing spline, escolhendo o nível de suavização com validação ou conhecimento da incerteza dos sensores.
4. **Extremidades:** avaliaria os erros separadamente perto de $x=0$ e $x=6$, pois polinômios globais podem oscilar nessas regiões. A distribuição de Chebyshev concentra nós nas bordas. Também evitaria extrapolar sem um modelo físico adicional.
5. **Temperatura verdadeira desconhecida:** em campo, não é possível calcular diretamente o erro verdadeiro, pois não se conhece $T(x)$. Usaria validação cruzada (retirar sensores temporariamente e prever suas leituras), repetição de medições, análise dos resíduos e conhecimento físico do processo.

**Métricas utilizadas:** erro máximo absoluto e RMSE (além do MAE na etapa 1). O erro máximo identifica o pior desvio; o RMSE penaliza mais erros grandes. A interpolação dentro do intervalo $[0,6]$ é interpolação; estimar fora dele é extrapolação e exige cautela, pois os métodos não têm garantia de representar o comportamento físico além dos sensores.

**Conclusão:** para dados exatos e poucos sensores, Lagrange pode ser competitivo. Com muitos sensores, splines ou Lagrange com nós de Chebyshev tendem a oferecer maior estabilidade que Lagrange equiespaçado. Com ruído, é preferível uma abordagem de suavização ou regressão em vez de interpolar exatamente cada leitura. A escolha final deve ser baseada nas métricas, na distribuição dos sensores e na incerteza experimental.